# 32/ News Category Classifier

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/rmisra/news-category-dataset](https://www.kaggle.com/datasets/rmisra/news-category-dataset)

> ~210k HuffPost news headlines (2012-2022) labeled with one of 42 categories (POLITICS, WELLNESS, ENTERTAINMENT, etc.) in a JSON-lines file.

**Goal:** Classify a news headline into its topic category.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_style("whitegrid")


In [ ]:
df = pd.read_json("data/News_Category_Dataset_v3.json", lines=True)
df['content'] = df['headline'].astype(str) + " " + df['short_description'].astype(str)
# Optionally collapse to top-N most frequent categories for a faster/cleaner demo
top_categories = df['category'].value_counts().head(10).index
df = df[df['category'].isin(top_categories)].reset_index(drop=True)
print(df.shape)
df['category'].value_counts()


## Exploratory Data Analysis

In [ ]:
print(df['category'].value_counts())
sns.countplot(y='category', data=df, order=df['category'].value_counts().index)
plt.title("Label distribution")
plt.show()

df['text_length'] = df['content'].astype(str).apply(len)
df['text_length'].hist(bins=50)
plt.title("Text length distribution")
plt.show()


## Text Cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['content'].apply(clean_text)
df[['content', 'clean_text']].head()


## Train/Test Split & TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(X_train_tfidf.shape)


## Model Training

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)


## Evaluation

In [ ]:
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## Try It Yourself

In [ ]:
def predict_sample(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    return model.predict(vec)[0]

predict_sample("Replace this string with your own example text")


## Next Steps
- Try `TfidfVectorizer` + `LinearSVC`, or fine-tune a transformer (`DistilBERT`) with 🤗 `transformers` for higher accuracy.
- Add lemmatization/stopword tuning.
- Track experiments with cross-validation instead of a single split.